# **UTS TEKNIK PENGAMBILAN SAMPEL & DATA WRANGLING**
* **NAMA : GAY FAUSTINE**
* **NIM  : 241061014**






In [ ]:
!pip install requests beautifulsoup4

# Import Library

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

# 1. Melakukan Scraping

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

BASE_URL = "https://myhartono.com/en/android/page-{}/"
List_Android = []

for page in range(1, 4):
    url = BASE_URL.format(page)
    print(f'\n🔍 Scraping halaman {page}: {url}')

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        titles = soup.find_all("a", class_="product-title")
        prices = soup.find_all("span", class_="ty-price-num")

        print(f'   Produk ditemukan: {len(titles)}')

        for i, (title, price) in enumerate(zip(titles, prices), start=1):
            item = {
                "no"      : len(List_Android) + 1,
                "halaman" : page,
                "title"   : title.get_text(strip=True),
                "harga"   : price.get_text(strip=True)
            }
            List_Android.append(item)
            print(f'   [{i}] {item["title"]} — {item["harga"]}')

    except Exception as e:
        print(f' Error halaman {page}: {e}')

    time.sleep(1.5)

print(f'\n Total produk: {len(List_Android)}')


🔍 Scraping halaman 1: https://myhartono.com/en/android/page-1/
   Produk ditemukan: 24
   [1] SAMSUNG SMARTPHONE GALAXY A57 5G SERIES — Rp
   [2] SAMSUNG SMARTPHONE GALAXY A37 SERIES — 7.109.000
   [3] SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES — Rp
   [4] SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES — 6.109.000
   [5] SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES — Rp
   [6] SAMSUNG SMARTPHONE GALAXY S25+ SERIES — 28.509.000
   [7] SAMSUNG SMARTPHONE GALAXY S25 SERIES — Rp
   [8] SAMSUNG SMARTPHONE GALAXY S25 FE SERIES — 18.009.000
   [9] SAMSUNG SMARTPHONE GALAXY A56 5G SERIES — Rp
   [10] SAMSUNG SMARTPHONE GALAXY A36 5G SERIES — 17.109.000
   [11] SAMSUNG SMARTPHONE GALAXY A26 5G 8/265GB SERIES — Rp
   [12] SAMSUNG SMARTPHONE GALAXY A17 5G SERIES — 15.009.000
   [13] SAMSUNG SMARTPHONE GALAXY A17 SERIES — Rp
   [14] SAMSUNG SMARTPHONE GALAXY A07 SERIES — 11.809.000
   [15] OPPO SMARTPHONE RENO 15 PRO MAX SERIES — Rp
   [16] OPPO SMARTPHONE RENO 15 5G SERIES — 11.009.000
   [17] OPPO 

Berdasarkan hasil scraping di atas, terlihat bahwa produk pada nomor ganjil seperti 1, 3, 5, 7 dan seterusnya tidak memiliki harga, sementara produk nomor genap memiliki harga. Pola ini terjadi bukan karena harga produknya memang kosong di website, melainkan karena cara pengambilan data harga yang kurang tepat.

Masalah utamanya ada pada penggunaan soup.find_all("span", class_="ty-price-num") yang mengambil semua elemen harga sekaligus dari seluruh halaman. Di website myhartono.com, setiap produk kemungkinan memiliki dua elemen harga dalam HTML-nya, yaitu harga normal dan harga coret atau harga lama, yang keduanya menggunakan class yang sama yaitu ty-price-num. Akibatnya jumlah elemen harga yang ditemukan menjadi dua kali lipat dari jumlah produk, yaitu 48 harga untuk 24 produk.

Ketika zip(titles, prices) dijalankan, produk pertama dipasangkan dengan elemen harga pertama yang ternyata adalah harga kosong atau harga coret milik produk itu sendiri, sedangkan harga aslinya malah berpindah dan dipasangkan ke produk kedua. Hal ini terus berulang sehingga semua produk bernomor ganjil mendapat harga kosong dan produk bernomor genap mendapat harga yang seharusnya milik produk sebelumnya.

# 2. Mengubah File Menjadi Format JSON

In [24]:
with open("List_Android.json", "w", encoding="utf-8") as f:
    json.dump(List_Android, f, ensure_ascii=False, indent=2)

print("File List_Android.json berhasil disimpan")

File List_Android.json berhasil disimpan


# 3. Import Library ETL

In [25]:
import pandas as pd
import re

# a. Extract

In [26]:
with open("List_Android.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(df.head())

   no  halaman                                        title      harga
0   1        1      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES         Rp
1   2        1         SAMSUNG SMARTPHONE GALAXY A37 SERIES  7.109.000
2   3        1  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES         Rp
3   4        1  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES  6.109.000
4   5        1   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES         Rp


# b. Transform

In [31]:
def bersihkan_harga(h):
    angka = re.sub(r"[^\d]", "", str(h))
    return int(angka) if angka else 0

df["harga"] = df["harga"].apply(bersihkan_harga)
df["title"] = df["title"].str.strip()

print(df.head(10))

   no  halaman                                        title     harga
0   1        1      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES         0
1   2        1         SAMSUNG SMARTPHONE GALAXY A37 SERIES   7109000
2   3        1  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES         0
3   4        1  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES   6109000
4   5        1   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES         0
5   6        1        SAMSUNG SMARTPHONE GALAXY S25+ SERIES  28509000
6   7        1         SAMSUNG SMARTPHONE GALAXY S25 SERIES         0
7   8        1      SAMSUNG SMARTPHONE GALAXY S25 FE SERIES  18009000
8   9        1      SAMSUNG SMARTPHONE GALAXY A56 5G SERIES         0
9  10        1      SAMSUNG SMARTPHONE GALAXY A36 5G SERIES  17109000


# c. Load

In [32]:
df.to_csv("List_Android.csv", index=False, encoding="utf-8-sig")
print("File List_Android.csv berhasil disimpan")

File List_Android.csv berhasil disimpan
